# Bibliographic references from OpenAlex

This notebook downloads a small OpenAlex sample and loads it into both
`NetworkXGraph` and `Neo4jGraph`.

Dataset notes:
- papers from an OpenAlex search query
- author nodes created from authorships
- citation links are added when both papers are present in the sample
- if network access is restricted, the loader falls back to bundled demo data

In [ ]:
import importlib.util

# Comprovació de presència del paquet
package_to_check = 'drm'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade drm-tools
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


In [ ]:
import os

from drm import Neo4jGraph, NetworkXGraph
from drm.exemples import load_bibliografia_openalex


def neo4j_config(default_target="DEV"):
    target = os.environ.get("NEO4J_TARGET", default_target).upper()
    prefix = f"NEO4J_{target}_"
    return {
        "target": target,
        "url": os.environ.get(f"{prefix}URL", os.environ.get("NEO4J_URL", "bolt://localhost:7687")),
        "user": os.environ.get(f"{prefix}USER", os.environ.get("NEO4J_USER", "neo4j")),
        "password": os.environ.get(f"{prefix}PASSWORD", os.environ.get("NEO4J_PASSWORD", "secret")),
        "database": os.environ.get(f"{prefix}DATABASE", os.environ.get("NEO4J_DATABASE", "neo4j")),
    }

## NetworkXGraph

In [ ]:
nx_graph = NetworkXGraph()
nx_stats = load_bibliografia_openalex(nx_graph, query="graph database", per_page=15)
print("NetworkX stats:", nx_stats)

# Show paper data (not just raw IDs)
node_ids = nx_graph.find_nodes_by_property("source", "openalex")[:5]
for nid in node_ids:
    attrs = nx_graph.get_node_attrs(nid)
    print(f"  {attrs.get('main_label', '?')}: {attrs.get('title', attrs.get('name', nid))}")

nx_graph.close()

## Neo4jGraph

In [ ]:
cfg = neo4j_config()
print("Using Neo4j target:", cfg["target"])
neo4j_graph = Neo4jGraph(
    url=cfg["url"],
    user=cfg["user"],
    password=cfg["password"],
    database=cfg["database"],
)
try:
    neo4j_stats = load_bibliografia_openalex(neo4j_graph, query="graph database", per_page=15)
    print("Neo4j stats:", neo4j_stats)
finally:
    neo4j_graph.close()
